Reverses changes made by decompartmentalize.ipynb

In [ ]:
# Consider reviewing spontaneous rxns identified by this process and adding them back into the original model, as these may occur in vivo in their newly added compartment
import cobra,json,os,sys,shutil,re,pandas as pd, io, requests
from bs4 import BeautifulSoup
from collections import OrderedDict
from copy import deepcopy

# keep going up until the "pycore" folder is found
wd = os.getcwd()
while not os.path.exists('pycore'):
	os.chdir('..')
base_path = os.getcwd()
sys.path.append(base_path)
from pycore import *

os.chdir(wd)

/Users/ejm6426/Documents/pkRBA/pycore/../build_model/input/iIsor850_RBA.json


In [2]:
# change all protein locations to exclude subloc_decompartmentalized values
pro_path = 'input/PROTEIN_stoich_curation.tsv'
decomp_dir = 'input/decompartmentalized'
enz_path = 'input/ENZYME_stoich.tsv'
if not os.path.isdir(decomp_dir):
	os.mkdir(decomp_dir)
shutil.copy(pro_path,os.path.join(decomp_dir, 'PROTEIN_stoich_curation.tsv'))
shutil.copy(enz_path,os.path.join(decomp_dir, 'ENZYME_stoich.tsv'))

df_pro = read_spreadsheet(pro_path)
# remove subloc_assigned values from subloc_decompartmentalized
for i in df_pro.index:
	if pd.notna(df_pro.loc[i,"subloc_decompartmentalized"]):
		df_pro.at[i,"subloc_assigned"] = ','.join([r for r in df_pro.loc[i,"subloc_assigned"].split(',') if r not in df_pro.loc[i,"subloc_decompartmentalized"].split(',')])
	PTM_stoich_edits = {}
	PTM_stoich_edits_final = {}
	if pd.notna(df_pro.loc[i,"PTM_stoich_edits"]):
		if df_pro.loc[i, "PTM_stoich_edits"].strip().startswith("{") and df_pro.loc[i, "PTM_stoich_edits"].strip().endswith("}"):
			PTM_stoich_edits = eval(df_pro.loc[i,"PTM_stoich_edits"])
		else:
			PTM_stoich_edits["default"] = df_pro.loc[i,"PTM_stoich_edits"]
		for k,v in PTM_stoich_edits.items():
			condition_options = []
			for d in v.split('|'):
				edit_dict = {}
				for pair in d.split(','):
					key, value = pair.split(':')
					if key not in model.metabolites:
						continue
					edit_dict[key] = value
				if len(edit_dict) > 0:
					condition_options.append(edit_dict)
			if len(condition_options) > 0:
				PTM_stoich_edits_final[k] = '|'.join([','.join([f"{k}:{v}" for k,v in edit.items()]) for edit in condition_options])
		if len(PTM_stoich_edits.keys()) == 1 and "default" in PTM_stoich_edits.keys():
			PTM_stoich_edits_final = PTM_stoich_edits_final["default"]
		df_pro.at[i,"PTM_stoich_edits"] = str(PTM_stoich_edits_final)
# save w/o index column
df_pro.to_csv(pro_path, sep='\t', index=False)

In [3]:
# copy model into decompartmentalized folder
shutil.copy('input/model_decompartmentalized.json',os.path.join(decomp_dir, os.path.basename(SM_path)))

'input/decompartmentalized/iIsor850_RBA.json'

In [4]:
#remove all rxns from enz sheet that aren't in GSM model
df_enz = read_spreadsheet(enz_path)
for i in df_enz.index:
	df_enz.at[i, "rxns"] = ','.join([r for r in df_enz.loc[i, "rxns"].split(',') if r in model.reactions])
	if len(df_enz.loc[i,"rxns"]) == 0:
		df_enz.drop(i, inplace=True)
df_enz.to_csv(enz_path, sep='\t', index=False)